# Laboratorio 1.2 · Ingesta desde archivos y APIs

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** la analítica de una tienda en línea

Hoy construimos la **capa de ingesta** del caso integrador con cuatro fuentes:

| Parte | Fuente | Lo que practicas |
|---|---|---|
| 1 | Archivos CSV de tres proveedores | Codificaciones, delimitadores, contratos, cuarentena, idempotencia |
| 2 | API de catálogo (DummyJSON) y tipo de cambio (Frankfurter) | Paginación por offset, rangos de fechas, reintentos declarativos |
| 3 | **API caprichosa** de pedidos (corre dentro de este notebook) | Secretos, 401 vs 429 vs 5xx, backoff con jitter, cursor, checkpoint |
| 4 | Los logs de todo lo anterior | Logs estructurados como datos |

Todo aterriza en una **landing zone** con la misma estructura que usarías en Cloud Storage o S3:

```
landing/
├── raw/          lo que llegó, tal cual
├── bronze/       Parquet con metadatos, particionado por fecha_carga=
├── _cuarentena/  filas que no cumplieron el contrato
├── _manifiesto.jsonl
└── _logs/
```

> **Antes de empezar:** abre el panel de **Secretos** de Colab (ícono de llave en la barra izquierda), crea un secreto llamado
> `API_KEY_CAPRICHOSA` con el valor `bsg-demo-2026` y activa el acceso para este notebook.

## Preparación

In [ ]:
import json, os, logging
from datetime import date
from pathlib import Path

import pandas as pd
import requests

LANDING = Path("landing")
ENTRADA = Path("entrada/proveedores")
FECHA_CARGA = date.today().isoformat()
MODO_OFFLINE = False      # True = catálogo y tipo de cambio desde los espejos locales (sin internet)
PUERTO = 8765

print("Fecha de carga:", FECHA_CARGA)

### Los módulos del kit

Las siguientes celdas contienen **exactamente el mismo código** que el kit local que usarás de tarea
(`ingesta/logs.py`, `ingesta/cliente_http.py`, etc.). Están colapsadas: ejecútalas y ábrelas cuando la explicación las mencione.

In [ ]:
# @title logs.py · logging estructurado en JSON
"""Logging estructurado: cada evento es una línea JSON que se puede consultar como datos."""
import json
import logging
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

RUN_ID = uuid.uuid4().hex[:12]   # identifica una ejecución completa del pipeline


class FormatoJSON(logging.Formatter):
    def format(self, registro):
        linea = {
            "ts": datetime.fromtimestamp(registro.created, tz=timezone.utc).isoformat(timespec="milliseconds"),
            "nivel": registro.levelname,
            "evento": registro.getMessage(),
            "run_id": RUN_ID,
        }
        linea.update(getattr(registro, "campos", {}))
        if registro.exc_info:
            linea["error"] = self.formatException(registro.exc_info).splitlines()[-1]
        return json.dumps(linea, ensure_ascii=False, default=str)


def configurar_logs(archivo=None, nivel="INFO"):
    """Envía los eventos a la consola y, opcionalmente, a un archivo .jsonl."""
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except (AttributeError, ValueError):
        pass
    log = logging.getLogger("ingesta")
    log.setLevel(nivel)
    log.handlers.clear()
    log.propagate = False
    consola = logging.StreamHandler(sys.stdout)
    consola.setFormatter(FormatoJSON())
    log.addHandler(consola)
    if archivo:
        Path(archivo).parent.mkdir(parents=True, exist_ok=True)
        disco = logging.FileHandler(archivo, encoding="utf-8")
        disco.setFormatter(FormatoJSON())
        log.addHandler(disco)
    return log


def evento(nombre, nivel=logging.INFO, **campos):
    """Registra un evento con campos estructurados: evento("archivo_ingestado", filas=120)."""
    logging.getLogger("ingesta").log(nivel, nombre, extra={"campos": campos})

configurar_logs(LANDING / "_logs" / f"lab_{FECHA_CARGA}.jsonl")
evento("laboratorio_iniciado", curso="pipelines", sesion="1.2")

In [ ]:
# @title cliente_http.py · reintentos con backoff, jitter y Retry-After
"""Clientes HTTP robustos: timeouts, reintentos con backoff exponencial y jitter, y Retry-After."""
import logging
import random
import time

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# 408 y 425: la solicitud puede repetirse; 429: límite de tasa; 5xx: problema del servidor
REINTENTABLES = {408, 425, 429, 500, 502, 503, 504}


class ErrorNoReintentable(Exception):
    """El servidor dijo que la solicitud está mal (400, 401, 403, 404...). Reintentar no ayuda."""


class ReintentosAgotados(Exception):
    """Se intentó varias veces y el problema temporal no se resolvió."""


def sesion_con_retry(max_intentos=5, backoff=0.5):
    """Opción declarativa: urllib3 reintenta por nosotros. Ideal para APIs bien portadas."""
    politica = Retry(total=max_intentos, backoff_factor=backoff, status_forcelist=sorted(REINTENTABLES),
                     allowed_methods=["GET"], respect_retry_after_header=True)
    sesion = requests.Session()
    sesion.mount("https://", HTTPAdapter(max_retries=politica))
    sesion.mount("http://", HTTPAdapter(max_retries=politica))
    sesion.headers["User-Agent"] = "bsg-curso-pipelines/1.2"
    return sesion


def espera_backoff(intento, base=0.5, tope=20.0, rng=random):
    """Backoff exponencial con 'full jitter': un valor al azar entre 0 y min(tope, base * 2^intento)."""
    return rng.uniform(0, min(tope, base * 2 ** intento))


def _retry_after(respuesta):
    valor = respuesta.headers.get("Retry-After", "")
    try:
        return max(0.0, float(valor))
    except ValueError:
        return None   # también puede venir como fecha HTTP; en ese caso usamos backoff


def get_con_reintentos(sesion, url, *, params=None, headers=None, max_intentos=6,
                       timeout=(3.05, 10), base=0.5, tope=20.0, rng=random):
    """GET que distingue errores temporales (reintenta) de errores definitivos (falla de inmediato)."""
    motivo = "sin intentos"
    for intento in range(1, max_intentos + 1):
        inicio = time.perf_counter()
        try:
            r = sesion.get(url, params=params, headers=headers, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as e:
            motivo = type(e).__name__
            espera = espera_backoff(intento, base, tope, rng)
        else:
            ms = round((time.perf_counter() - inicio) * 1000)
            if r.status_code < 400:
                evento("http_ok", url=url, params=params, status=r.status_code, intento=intento, ms=ms)
                return r
            if r.status_code not in REINTENTABLES:
                evento("http_error_definitivo", logging.ERROR, url=url, status=r.status_code, cuerpo=r.text[:200])
                raise ErrorNoReintentable(f"HTTP {r.status_code} en {url}: {r.text[:200]}")
            motivo = f"HTTP {r.status_code}"
            ra = _retry_after(r)
            espera = ra if ra is not None else espera_backoff(intento, base, tope, rng)
        if intento == max_intentos:
            break
        evento("http_reintento", logging.WARNING, url=url, motivo=motivo, intento=intento, espera_s=round(espera, 2))
        time.sleep(espera)
    evento("http_reintentos_agotados", logging.ERROR, url=url, intentos=max_intentos, ultimo_motivo=motivo)
    raise ReintentosAgotados(f"{url}: {max_intentos} intentos sin éxito (último motivo: {motivo})")

In [ ]:
# @title archivos.py · contratos, landing zone, cuarentena y manifiesto
"""Ingesta de archivos: detección, contratos por proveedor, landing zone, cuarentena y manifiesto."""
import csv
import hashlib
import json
import logging
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from charset_normalizer import from_path


COLUMNAS_INVENTARIO = ["sku", "descripcion", "cantidad", "precio_unitario", "moneda", "fecha_corte"]


class ErrorCalidad(Exception):
    """Demasiadas filas rechazadas: el archivo no se publica en bronze."""


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 16), b""):
            h.update(bloque)
    return h.hexdigest()


def escribir_atomico(destino, escribir):
    """Escribe en un temporal y lo renombra: nadie ve nunca un archivo a medias."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = destino.with_name(destino.name + ".tmp")
    escribir(temporal)
    os.replace(temporal, destino)


def detectar(ruta, muestra=20000):
    """Sugerencia automática de codificación y delimitador. Sirve para DESCUBRIR, no para OPERAR."""
    mejor = from_path(ruta).best()
    codificacion = mejor.encoding if mejor else "utf-8"
    with open(ruta, "rb") as f:
        texto = f.read(muestra).decode(codificacion, errors="replace")
    try:
        delimitador = csv.Sniffer().sniff(texto, delimiters=",;|\t").delimiter
    except csv.Error:
        delimitador = None
    return {"archivo": Path(ruta).name, "codificacion": codificacion, "delimitador": delimitador,
            "bom_utf8": texto.startswith("﻿"), "primera_linea": texto.lstrip("﻿").splitlines()[0][:80]}


def leer_con_contrato(ruta, contrato):
    """Lee un CSV aplicando el contrato del proveedor. Devuelve (DataFrame canónico, filas rechazadas)."""
    rechazadas = []

    def fila_mala(campos):
        rechazadas.append(contrato["sep"].join(campos))
        return None   # None = no incluir la fila, pero ya quedó registrada

    df = pd.read_csv(ruta, encoding=contrato["codificacion"], sep=contrato["sep"],
                     decimal=contrato.get("decimal", "."), thousands=contrato.get("miles"),
                     # el SKU siempre como texto: "00123" no es el número 123
                     dtype={origen: "string" for origen, canon in contrato["columnas"].items() if canon == "sku"},
                     engine="python", on_bad_lines=fila_mala)
    faltantes = set(contrato["columnas"]) - set(df.columns)
    if faltantes:
        raise ErrorCalidad(f"{Path(ruta).name}: faltan columnas del contrato {sorted(faltantes)}")
    df = df.rename(columns=contrato["columnas"])[COLUMNAS_INVENTARIO]
    df["sku"] = df["sku"].astype("string").str.strip()
    df["descripcion"] = df["descripcion"].astype("string").str.strip()
    df["moneda"] = df["moneda"].astype("string").str.strip().str.upper()
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").astype("Int64")
    df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")
    df["fecha_corte"] = pd.to_datetime(df["fecha_corte"], format=contrato["formato_fecha"], errors="coerce").dt.date
    # Filas que se leyeron pero traen valores inválidos también van a cuarentena
    invalidas = df["cantidad"].isna() | df["precio_unitario"].isna() | df["fecha_corte"].isna() | df["sku"].isna()
    for _, fila in df[invalidas].iterrows():
        rechazadas.append("valores inválidos: " + json.dumps(fila.astype(str).to_dict(), ensure_ascii=False))
    return df[~invalidas].reset_index(drop=True), rechazadas


class Manifiesto:
    """Registro de todo lo que se ha ingestado. Permite que la ingesta sea idempotente."""

    def __init__(self, landing):
        self.ruta = Path(landing) / "_manifiesto.jsonl"

    def entradas(self):
        if not self.ruta.exists():
            return []
        return [json.loads(l) for l in self.ruta.read_text(encoding="utf-8").splitlines() if l.strip()]

    def ya_ingestado(self, hash_archivo):
        return any(e.get("hash") == hash_archivo and e.get("estado") == "ok" for e in self.entradas())

    def registrar(self, **entrada):
        self.ruta.parent.mkdir(parents=True, exist_ok=True)
        with open(self.ruta, "a", encoding="utf-8") as f:
            f.write(json.dumps(entrada, ensure_ascii=False, default=str) + "\n")


def aterrizar_archivo(ruta, fuente, contrato, landing, fecha_carga, max_rechazo=0.05):
    """raw (copia exacta) -> lectura con contrato -> cuarentena -> bronze (Parquet) -> manifiesto."""
    ruta, landing = Path(ruta), Path(landing)
    manifiesto = Manifiesto(landing)
    h = sha256(ruta)
    if manifiesto.ya_ingestado(h):
        evento("archivo_omitido_ya_ingestado", fuente=fuente, archivo=ruta.name, hash=h[:12])
        return None

    particion = f"fecha_carga={fecha_carga}"
    destino_raw = landing / "raw" / "proveedores" / fuente / particion / ruta.name
    escribir_atomico(destino_raw, lambda tmp: shutil.copyfile(ruta, tmp))   # tal cual llegó

    df, rechazadas = leer_con_contrato(ruta, contrato)
    total = len(df) + len(rechazadas)
    tasa = len(rechazadas) / total if total else 0.0
    if rechazadas:
        destino_q = landing / "_cuarentena" / "proveedores" / fuente / particion / f"{h[:12]}.txt"
        escribir_atomico(destino_q, lambda tmp: Path(tmp).write_text("\n".join(rechazadas), encoding="utf-8"))
        evento("filas_en_cuarentena", logging.WARNING, fuente=fuente, archivo=ruta.name, filas=len(rechazadas), tasa=round(tasa, 3))
    if tasa > max_rechazo:
        manifiesto.registrar(hash=h, fuente=fuente, archivo=ruta.name, estado="rechazado", filas=len(df),
                             rechazadas=len(rechazadas), run_id=RUN_ID, ts=datetime.now(timezone.utc))
        raise ErrorCalidad(f"{ruta.name}: {tasa:.1%} de filas rechazadas supera el máximo de {max_rechazo:.0%}")

    df["_fuente"] = fuente
    df["_archivo"] = ruta.name
    df["_hash_archivo"] = h
    df["_cargado_en"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    df["_run_id"] = RUN_ID
    destino_bronze = landing / "bronze" / "inventario" / particion / f"fuente={fuente}" / f"{h[:12]}.parquet"
    escribir_atomico(destino_bronze, lambda tmp: df.to_parquet(tmp, index=False))

    manifiesto.registrar(hash=h, fuente=fuente, archivo=ruta.name, estado="ok", filas=len(df),
                         rechazadas=len(rechazadas), bytes=ruta.stat().st_size, destino=str(destino_bronze),
                         run_id=RUN_ID, ts=datetime.now(timezone.utc))
    evento("archivo_ingestado", fuente=fuente, archivo=ruta.name, filas=len(df), rechazadas=len(rechazadas))
    return df

In [ ]:
# @title apis.py · offset, rango de fechas y cursor con checkpoint
"""Ingesta desde APIs: paginación por offset (catálogo), rango de fechas (tipo de cambio) y cursor (pedidos)."""
import json
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import pandas as pd



def _guardar_json(destino, cuerpo):
    escribir_atomico(destino, lambda tmp: Path(tmp).write_text(json.dumps(cuerpo, ensure_ascii=False), encoding="utf-8"))


def _metadatos(df, fuente):
    df["_fuente"] = fuente
    df["_cargado_en"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    df["_run_id"] = RUN_ID
    return df


def catalogo(sesion, base_url, landing, fecha_carga, tamano_pagina=30):
    """Paginación por offset (limit/skip). La sesión ya trae reintentos declarativos (urllib3)."""
    particion = Path(landing) / "raw" / "catalogo" / f"fecha_carga={fecha_carga}"
    productos, salto, pagina, total = [], 0, 0, None
    while True:
        r = sesion.get(f"{base_url}/products", timeout=(3.05, 15),
                       params={"limit": tamano_pagina, "skip": salto, "select": "title,category,price,stock,brand,sku"})
        r.raise_for_status()
        cuerpo = r.json()
        _guardar_json(particion / f"pagina_{pagina:04d}.json", cuerpo)   # raw: la respuesta tal cual
        total = cuerpo["total"]
        productos.extend(cuerpo["products"])
        evento("pagina_descargada", fuente="catalogo", pagina=pagina, registros=len(cuerpo["products"]), acumulado=len(productos), total=total)
        salto += len(cuerpo["products"])
        pagina += 1
        if not cuerpo["products"] or salto >= total:
            break
    df = pd.DataFrame(productos)
    for col in ("title", "category", "price", "stock", "brand", "sku"):
        if col not in df:
            df[col] = None   # algunos productos no traen todos los campos
    if len(df) != total or not df["id"].is_unique:
        raise ValueError(f"Catálogo incompleto o duplicado: {len(df)} filas, {df['id'].nunique()} únicas, total esperado {total}")
    destino = Path(landing) / "bronze" / "catalogo" / f"fecha_carga={fecha_carga}" / "catalogo.parquet"
    escribir_atomico(destino, lambda tmp: _metadatos(df, "dummyjson").to_parquet(tmp, index=False))
    evento("fuente_completa", fuente="catalogo", filas=len(df), paginas=pagina)
    return df


def tipo_cambio(sesion, base_url, landing, fecha_carga, base="USD", simbolos=("MXN", "BRL", "EUR"), dias=30):
    """Una sola llamada por rango de fechas; aplanamos {fecha: {moneda: tasa}} a filas."""
    hasta = date.fromisoformat(str(fecha_carga))
    desde = hasta - timedelta(days=dias)
    r = sesion.get(f"{base_url}/{desde}..{hasta}", params={"base": base, "symbols": ",".join(simbolos)}, timeout=(3.05, 15))
    r.raise_for_status()
    cuerpo = r.json()
    _guardar_json(Path(landing) / "raw" / "tipo_cambio" / f"fecha_carga={fecha_carga}" / "respuesta.json", cuerpo)
    filas = [{"fecha": f, "moneda_base": cuerpo["base"], "moneda": m, "tasa": t}
             for f, tasas in cuerpo["rates"].items() for m, t in tasas.items()]
    df = pd.DataFrame(filas)
    destino = Path(landing) / "bronze" / "tipo_cambio" / f"fecha_carga={fecha_carga}" / "tipo_cambio.parquet"
    escribir_atomico(destino, lambda tmp: _metadatos(df, "frankfurter").to_parquet(tmp, index=False))
    evento("fuente_completa", fuente="tipo_cambio", filas=len(df), desde=str(desde), hasta=str(hasta))
    return df


def pedidos(sesion, base_url, api_key, landing, fecha_carga, limite=50, reanudar=True, **opciones_http):
    """Paginación por cursor, con checkpoint: si falla a la mitad, la siguiente corrida continúa donde se quedó."""
    landing = Path(landing)
    raw = landing / "raw" / "pedidos" / f"fecha_carga={fecha_carga}"
    checkpoint = landing / "_checkpoints" / f"pedidos_{fecha_carga}.json"
    cursor, pagina = None, 0
    if reanudar and checkpoint.exists():
        estado = json.loads(checkpoint.read_text(encoding="utf-8"))
        cursor, pagina = estado["cursor"], estado["pagina"]
        evento("reanudando_desde_checkpoint", fuente="pedidos", pagina=pagina)
    total_esperado = None
    while True:
        params = {"limit": limite, **({"cursor": cursor} if cursor else {})}
        r = get_con_reintentos(sesion, f"{base_url}/v1/pedidos", params=params,
                               headers={"X-API-Key": api_key}, **opciones_http)
        cuerpo = r.json()
        total_esperado = int(r.headers.get("X-Total-Count", 0)) or total_esperado
        escribir_atomico(raw / f"pagina_{pagina:04d}.jsonl",
                         lambda tmp: Path(tmp).write_text("".join(json.dumps(p, ensure_ascii=False) + "\n" for p in cuerpo["data"]), encoding="utf-8"))
        cursor, pagina = cuerpo["next_cursor"], pagina + 1
        if cursor is None:
            break
        escribir_atomico(checkpoint, lambda tmp: Path(tmp).write_text(json.dumps({"cursor": cursor, "pagina": pagina}), encoding="utf-8"))
    # Consolidamos todas las páginas crudas (incluidas las de corridas anteriores) y validamos contra el total
    df = pd.concat([pd.read_json(p, lines=True) for p in sorted(raw.glob("pagina_*.jsonl"))], ignore_index=True)
    df = df.drop_duplicates(subset="id")
    if total_esperado and len(df) != total_esperado:
        raise ValueError(f"Pedidos incompletos: {len(df)} de {total_esperado}")
    destino = landing / "bronze" / "pedidos" / f"fecha_carga={fecha_carga}" / "pedidos.parquet"
    escribir_atomico(destino, lambda tmp: _metadatos(df, "api_caprichosa").to_parquet(tmp, index=False))
    checkpoint.unlink(missing_ok=True)
    evento("fuente_completa", fuente="pedidos", filas=len(df), paginas=pagina)
    return df

In [ ]:
# @title api_caprichosa.py · el servidor que falla a propósito
"""API caprichosa: un servidor local que se porta como las APIs reales en un mal día.

- /v1/pedidos      requiere X-API-Key, paginación por cursor, límite de tasa (429 + Retry-After),
                   errores 500/503 aleatorios y respuestas lentas.
- /dummyjson/...   espejo sin caos de dummyjson.com (para trabajar sin internet).
- /frankfurter/... espejo sin caos de api.frankfurter.dev/v1 (para trabajar sin internet).

Uso local:  python -m ingesta.api_caprichosa --puerto 8765 --tasa-error 0.15
"""
import argparse
import base64
import json
import random
import threading
import time
from datetime import date, timedelta
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

SEMILLA = 987654


class Estado:
    def __init__(self, api_key="bsg-demo-2026", tasa_error=0.15, prob_lento=0.05, segundos_lento=4.0,
                 peticiones_por_segundo=4, n_pedidos=500, semilla=SEMILLA):
        self.api_key, self.tasa_error, self.prob_lento = api_key, tasa_error, prob_lento
        self.segundos_lento, self.capacidad = segundos_lento, peticiones_por_segundo
        self.tokens, self.ultimo = float(peticiones_por_segundo), time.monotonic()
        self.rng, self.lock = random.Random(semilla), threading.Lock()
        datos = random.Random(semilla + 1)
        inicio = date.today() - timedelta(days=30)
        self.pedidos = [{
            "id": 100000 + i, "cliente_id": datos.randint(1, 800),
            "fecha": (inicio + timedelta(days=datos.randint(0, 29))).isoformat(),
            "total": round(datos.uniform(99, 25000), 2), "moneda": datos.choice(["MXN", "MXN", "MXN", "USD"]),
            "estado": datos.choice(["pagado", "enviado", "entregado", "cancelado"]), "articulos": datos.randint(1, 9),
        } for i in range(n_pedidos)]
        categorias = ["beauty", "fragrances", "furniture", "groceries", "home-decoration", "kitchen-accessories", "laptops"]
        self.productos = [{"id": i, "title": f"Producto sintético {i}", "category": datos.choice(categorias),
                           "price": round(datos.uniform(1, 2500), 2), "stock": datos.randint(0, 150),
                           "brand": datos.choice(["Acme", "Globex", None]), "sku": f"SIN-{i:05d}"} for i in range(1, 195)]

    def consumir_token(self):
        """Token bucket: se recargan `capacidad` tokens por segundo; sin token -> 429."""
        with self.lock:
            ahora = time.monotonic()
            self.tokens = min(self.capacidad, self.tokens + (ahora - self.ultimo) * self.capacidad)
            self.ultimo = ahora
            if self.tokens >= 1:
                self.tokens -= 1
                return True
            return False

    def tirar_dado(self):
        with self.lock:
            return self.rng.random(), self.rng.random(), self.rng.choice([500, 503])


def _cursor(offset):
    return base64.urlsafe_b64encode(json.dumps({"o": offset}).encode()).decode()


def _offset(cursor):
    return json.loads(base64.urlsafe_b64decode(cursor.encode()))["o"]


def crear_manejador(estado):
    class Manejador(BaseHTTPRequestHandler):
        def log_message(self, *args):   # silencio: el cliente es quien debe registrar
            pass

        def _json(self, codigo, cuerpo, encabezados=None):
            datos = json.dumps(cuerpo, ensure_ascii=False).encode("utf-8")
            self.send_response(codigo)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(datos)))
            for k, v in (encabezados or {}).items():
                self.send_header(k, str(v))
            self.end_headers()
            try:
                self.wfile.write(datos)
            except (BrokenPipeError, ConnectionResetError):
                pass   # el cliente se rindió (timeout) antes de que respondiéramos

        def do_GET(self):
            url = urlparse(self.path)
            q = {k: v[0] for k, v in parse_qs(url.query).items()}
            ruta = url.path.rstrip("/")
            if ruta == "/salud":
                return self._json(200, {"ok": True})
            if ruta == "/dummyjson/products":
                limite, salto = int(q.get("limit", 30)), int(q.get("skip", 0))
                return self._json(200, {"products": estado.productos[salto:salto + limite],
                                        "total": len(estado.productos), "skip": salto, "limit": limite})
            if ruta.startswith("/frankfurter/"):
                desde, hasta = ruta.split("/")[-1].split("..")
                simbolos = q.get("symbols", "MXN,BRL,EUR").split(",")
                d, h, tasas = date.fromisoformat(desde), date.fromisoformat(hasta), {}
                base = {"MXN": 17.0, "BRL": 5.1, "EUR": 0.86}
                rng = random.Random(SEMILLA)
                while d <= h:
                    if d.weekday() < 5:
                        tasas[d.isoformat()] = {s: round(base.get(s, 1.0) * rng.uniform(0.98, 1.02), 4) for s in simbolos}
                    d += timedelta(days=1)
                return self._json(200, {"amount": 1.0, "base": q.get("base", "USD"), "start_date": desde,
                                        "end_date": hasta, "rates": tasas})
            if ruta == "/v1/pedidos":
                if self.headers.get("X-API-Key") != estado.api_key:
                    return self._json(401, {"error": "API key inválida o ausente"})
                if not estado.consumir_token():
                    return self._json(429, {"error": "demasiadas solicitudes"}, {"Retry-After": 1})
                dado_error, dado_lento, codigo = estado.tirar_dado()
                if dado_error < estado.tasa_error:
                    return self._json(codigo, {"error": "falla temporal del servidor"})
                if dado_lento < estado.prob_lento:
                    time.sleep(estado.segundos_lento)
                limite = min(int(q.get("limit", 50)), 100)
                try:
                    salto = _offset(q["cursor"]) if q.get("cursor") else 0
                except Exception:
                    return self._json(400, {"error": "cursor inválido"})
                pagina = estado.pedidos[salto:salto + limite]
                siguiente = _cursor(salto + limite) if salto + limite < len(estado.pedidos) else None
                return self._json(200, {"data": pagina, "next_cursor": siguiente},
                                  {"X-Total-Count": len(estado.pedidos)})
            return self._json(404, {"error": f"ruta desconocida: {ruta}"})

    return Manejador


def iniciar(puerto=8765, **config):
    """Levanta el servidor en un hilo de fondo. Devuelve (servidor, estado)."""
    estado = Estado(**config)
    servidor = ThreadingHTTPServer(("127.0.0.1", puerto), crear_manejador(estado))
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    return servidor, estado

if "SERVIDOR" in globals():
    SERVIDOR.shutdown(); SERVIDOR.server_close()
SERVIDOR, ESTADO_API = iniciar(PUERTO)
print(f"API caprichosa lista en http://127.0.0.1:{PUERTO}")

In [ ]:
# @title generar_datos.py · archivos de proveedores con sus mañas
"""Genera los archivos de inventario de los proveedores, cada uno con sus propias mañas."""
import csv
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 987654
PRODUCTOS = ["Camión de juguete", "Muñeca de trapo", "Almohada de algodón", "Cafetera eléctrica",
             "Pañuelo de seda", "Lámpara de escritorio", "Cuaderno profesional, 100 hojas", "Termo de acero",
             "Sartén antiadherente", "Mochila escolar", "Balón de fútbol", "Audífonos inalámbricos",
             "Juego de té de cerámica", "Cepillo de dientes eléctrico", "Añejo de agave 750 ml"]


def _filas(rng, n, corte):
    for i in range(n):
        yield {
            "sku": f"{rng.randint(1, 999):05d}",          # con ceros a la izquierda
            "descripcion": rng.choice(PRODUCTOS),
            "cantidad": rng.randint(0, 2500),
            "precio": round(rng.uniform(15, 4800), 2),
            "moneda": rng.choice(["MXN", "USD"]),
            "fecha": corte,
        }


def _miles_coma(x):
    """1234.5 -> '1.234,50' (formato de muchos países de habla hispana)."""
    return f"{x:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")


def generar(carpeta, corte=None, semilla=SEMILLA):
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    rng = random.Random(semilla)
    corte = corte or (date.today() - timedelta(days=1))

    # Norte: el proveedor ordenado. UTF-8, coma, punto decimal, fecha ISO.
    with open(carpeta / "proveedor_norte.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["sku", "descripcion", "cantidad", "precio_unitario", "moneda", "fecha_corte"])
        for r in _filas(rng, 40, corte):
            w.writerow([r["sku"], r["descripcion"], r["cantidad"], f'{r["precio"]:.2f}', r["moneda"], r["fecha"].isoformat()])

    # Sur: Windows-1252, punto y coma, coma decimal y punto de miles, fecha dd/mm/aaaa, moneda en minúsculas.
    with open(carpeta / "proveedor_sur.csv", "w", encoding="cp1252", newline="") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow(["Código", "Descripción", "Existencia", "Precio Unit.", "Moneda", "Fecha"])
        for r in _filas(rng, 40, corte):
            w.writerow([r["sku"], r["descripcion"], r["cantidad"], _miles_coma(r["precio"]), f' {r["moneda"].lower()} ',
                        r["fecha"].strftime("%d/%m/%Y")])

    # Centro: UTF-8 con BOM, pipe, fecha estilo EE. UU. (mm/dd/aaaa), una fila con un campo de más y un "N/D".
    with open(carpeta / "proveedor_centro.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f, delimiter="|")
        w.writerow(["SKU", "DESCRIPCION", "QTY", "PRICE", "CURRENCY", "DATE"])
        for i, r in enumerate(_filas(rng, 45, corte)):
            cantidad = "N/D" if i == 17 else r["cantidad"]
            w.writerow([r["sku"], r["descripcion"], cantidad, f'{r["precio"]:.2f}', r["moneda"], r["fecha"].strftime("%m/%d/%Y")])
            if i == 30:
                f.write(f'99999|Producto misterioso|3|10.50|USD|{r["fecha"].strftime("%m/%d/%Y")}|CAMPO_EXTRA\r\n')

    # Oriente (reto): tabuladores, fecha dd-mm-aaaa y encabezados propios.
    with open(carpeta / "proveedor_oriente.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t")
        w.writerow(["item", "nombre", "unidades", "precio", "divisa", "corte"])
        for r in _filas(rng, 30, corte):
            w.writerow([r["sku"], r["descripcion"], r["cantidad"], f'{r["precio"]:.2f}', r["moneda"], r["fecha"].strftime("%d-%m-%Y")])

    return sorted(p.name for p in carpeta.glob("*.csv"))

## Parte 1 · Archivos de proveedores

Tres proveedores nos mandan su inventario. Cada uno lo hace a su manera. Veamos los primeros bytes de cada archivo:

In [ ]:
print(generar(ENTRADA))
for archivo in sorted(ENTRADA.glob("*.csv")):
    print(f"\n{archivo.name}\n  {open(archivo, 'rb').read(110)}")

Observa: `\xef\xbb\xbf` es el **BOM** de UTF-8; `\xf3` es la `ó` en **Windows-1252**; hay comas, punto y coma, *pipes* y tabuladores.

### El intento ingenuo

In [ ]:
try:
    pd.read_csv(ENTRADA / "proveedor_sur.csv")
except Exception as e:
    print("Falla 1 ->", type(e).__name__, ":", str(e)[:120])

try:
    df_mal = pd.read_csv(ENTRADA / "proveedor_sur.csv", encoding="latin-1")
    print("Falla 2 -> se leyó, pero en", df_mal.shape[1], "columna(s):", list(df_mal.columns)[:2])
except Exception as e:
    print("Falla 2 ->", type(e).__name__, ":", str(e)[:120])

# Falla 3: un archivo UTF-8 leído como Latin-1 no truena... pero corrompe el texto (mojibake)
print("\nFalla 3 -> sin error, pero mira las descripciones:")
pd.read_csv(ENTRADA / "proveedor_norte.csv", encoding="latin-1", dtype={"sku": str})[["sku", "descripcion"]].head(4)

Tres fallas distintas, y la tercera es la peor: no truena, corrompe en silencio. Leer "a ver si sale" no es ingesta. Necesitamos dos cosas:

1. **Detección** para *descubrir* cómo viene un archivo nuevo.
2. Un **contrato** explícito por proveedor para *operar* todos los días. Si el proveedor cambia algo, el contrato falla en voz alta.

In [ ]:
pd.DataFrame([detectar(p) for p in sorted(ENTRADA.glob("*.csv"))])

### Los contratos (en el kit viven en `config.yaml`)

In [ ]:
CONTRATOS = {
    "norte": {"codificacion": "utf-8", "sep": ",", "decimal": ".", "miles": None, "formato_fecha": "%Y-%m-%d",
              "columnas": {"sku": "sku", "descripcion": "descripcion", "cantidad": "cantidad",
                           "precio_unitario": "precio_unitario", "moneda": "moneda", "fecha_corte": "fecha_corte"}},
    "sur": {"codificacion": "cp1252", "sep": ";", "decimal": ",", "miles": ".", "formato_fecha": "%d/%m/%Y",
            "columnas": {"Código": "sku", "Descripción": "descripcion", "Existencia": "cantidad",
                         "Precio Unit.": "precio_unitario", "Moneda": "moneda", "Fecha": "fecha_corte"}},
    "centro": {"codificacion": "utf-8-sig", "sep": "|", "decimal": ".", "miles": None, "formato_fecha": "%m/%d/%Y",
               "columnas": {"SKU": "sku", "DESCRIPCION": "descripcion", "QTY": "cantidad",
                            "PRICE": "precio_unitario", "CURRENCY": "moneda", "DATE": "fecha_corte"}},
}

for fuente, contrato in CONTRATOS.items():
    aterrizar_archivo(ENTRADA / f"proveedor_{fuente}.csv", fuente, contrato, LANDING, FECHA_CARGA)

¿Qué quedó en la landing zone? ¿Y en cuarentena?

In [ ]:
def arbol(raiz, nivel=0, max_archivos=4):
    raiz = Path(raiz)
    hijos = sorted(raiz.iterdir())
    for i, h in enumerate(hijos):
        if h.is_file() and i >= max_archivos and nivel > 0:
            print("    " * (nivel + 1) + f"... ({len(hijos) - max_archivos} más)")
            break
        print("    " * nivel + ("📁 " if h.is_dir() else "   ") + h.name)
        if h.is_dir():
            arbol(h, nivel + 1, max_archivos)

arbol(LANDING)
for q in (LANDING / "_cuarentena").rglob("*.txt"):
    print(f"\nCuarentena: {q}\n{q.read_text(encoding='utf-8')}")

In [ ]:
inventario = pd.read_parquet(LANDING / "bronze" / "inventario")   # lee todas las particiones
print(inventario.groupby("fuente", observed=True).size().to_string(), "\n")
inventario.sample(5, random_state=987654)

Las columnas `fecha_carga` y `fuente` **no existen dentro de los archivos**: pandas las reconstruye a partir de las carpetas
`fecha_carga=.../fuente=...`. Eso es el particionado estilo *Hive*.

### ¿Parquet siempre ocupa menos?

In [ ]:
import time
chico = inventario.drop(columns=["fecha_carga", "fuente"])
grande = pd.concat([chico] * 5000, ignore_index=True)       # ~620 mil filas
for nombre, df in (("40 filas", chico.head(40)), (f"{len(grande):,} filas", grande)):
    df.to_csv("prueba.csv", index=False); df.to_parquet("prueba.parquet", index=False)
    t0 = time.perf_counter(); pd.read_csv("prueba.csv"); t_csv = time.perf_counter() - t0
    t0 = time.perf_counter(); pd.read_parquet("prueba.parquet", columns=["precio_unitario"]); t_pq = time.perf_counter() - t0
    print(f"{nombre:>14}: CSV {os.path.getsize('prueba.csv')/1024:9.1f} KB | Parquet {os.path.getsize('prueba.parquet')/1024:9.1f} KB"
          f" | leer 1 columna: CSV {t_csv:.3f} s vs Parquet {t_pq:.3f} s")
os.remove("prueba.csv"); os.remove("prueba.parquet")

Con pocos datos, los metadatos de Parquet pesan más que los datos. Con volumen gana por mucho, sobre todo si solo lees algunas columnas.

> **Ojo con el experimento:** replicamos las mismas 124 filas miles de veces, así que la compresión sale exagerada.
> Con datos reales la diferencia suele ser menor, pero sigue siendo grande. Nunca tomes un benchmark sin preguntar cómo se hizo.

## Parte 2 · APIs reales

- **Catálogo** (DummyJSON): paginación por `limit` y `skip` (offset). Cada página se guarda cruda en `raw/`.
- **Tipo de cambio** (Frankfurter, datos del Banco Central Europeo): una sola llamada por rango de fechas.

Aquí usamos `sesion_con_retry()`: urllib3 reintenta por nosotros. Es la opción declarativa, ideal para APIs que se portan bien.

In [ ]:
sesion = sesion_con_retry()
URL_CATALOGO = f"http://127.0.0.1:{PUERTO}/dummyjson" if MODO_OFFLINE else "https://dummyjson.com"
URL_TIPO_CAMBIO = f"http://127.0.0.1:{PUERTO}/frankfurter" if MODO_OFFLINE else "https://api.frankfurter.dev/v1"

df_catalogo = catalogo(sesion, URL_CATALOGO, LANDING, FECHA_CARGA, tamano_pagina=30)
df_catalogo.head()

In [ ]:
df_tc = tipo_cambio(sesion, URL_TIPO_CAMBIO, LANDING, FECHA_CARGA, base="USD", simbolos=("MXN", "BRL", "EUR"), dias=30)
df_tc.pivot(index="fecha", columns="moneda", values="tasa").tail()

> El Banco Central Europeo no publica el sol peruano (PEN). **Pregunta:** si el negocio lo necesita, ¿qué fuente usarías y qué implicaría mezclar dos fuentes de tipo de cambio?

## Parte 3 · La API caprichosa

La API de pedidos corre dentro de este notebook y se porta como una API real en un mal día:

| Situación | Respuesta | ¿Reintentar? |
|---|---|---|
| Sin API key o con una equivocada | `401` | **No**: reintentar no la arregla |
| Más de 4 solicitudes por segundo | `429` + `Retry-After: 1` | Sí, esperando lo que pide |
| Falla temporal (15 %) | `500` o `503` | Sí, con backoff |
| Respuesta lenta (5 %) | tarda 4 s | Sí: nuestro timeout de lectura es de 3 s |

### Primero, el secreto

In [ ]:
def obtener_secreto(nombre):
    # Orden: Secretos de Colab -> variable de entorno -> preguntar sin mostrar. Nunca en el código.
    try:
        from google.colab import userdata
        return userdata.get(nombre).strip()
    except Exception:
        pass
    if os.environ.get(nombre):
        return os.environ[nombre].strip()
    from getpass import getpass
    return getpass(f"Valor de {nombre} (no se mostrará): ").strip()

API_KEY = obtener_secreto("API_KEY_CAPRICHOSA")
BASE_PEDIDOS = f"http://127.0.0.1:{PUERTO}"
print("Secreto cargado:", API_KEY[:3] + "*" * (len(API_KEY) - 3))   # nunca imprimas un secreto completo

# Verificamos la llave antes de descargar nada (una sola página, con reintentos)
try:
    get_con_reintentos(requests.Session(), f"{BASE_PEDIDOS}/v1/pedidos", params={"limit": 1},
                       headers={"X-API-Key": API_KEY}, timeout=(3.05, 3))
    print("Llave verificada: la API la acepta.")
except ErrorNoReintentable:
    print("La API rechazó la llave (401). Revisa el secreto API_KEY_CAPRICHOSA en el panel de Secretos:"
          " nombre exacto, valor bsg-demo-2026 y acceso activado para este notebook. Luego vuelve a correr esta celda.")
except ReintentosAgotados:
    print("No hay conexión con la API caprichosa. Vuelve a ejecutar la celda de api_caprichosa.py y luego esta.")

### El cliente ingenuo

In [ ]:
pedidos_ingenuos, cursor = [], None
try:
    for pagina in range(20):
        params = {"limit": 50, **({"cursor": cursor} if cursor else {})}
        r = requests.get(f"{BASE_PEDIDOS}/v1/pedidos", params=params, headers={"X-API-Key": API_KEY})
        cuerpo = r.json()
        pedidos_ingenuos += cuerpo["data"]
        cursor = cuerpo["next_cursor"]
        if cursor is None:
            break
    print("¡Tuviste suerte! Sin fallas esta vez:", len(pedidos_ingenuos))
except Exception as e:
    print(f"El cliente ingenuo cayó en la página {pagina} con {len(pedidos_ingenuos)} pedidos.")
    print(f"HTTP {r.status_code} -> {type(e).__name__}: {e}  |  cuerpo: {r.text[:80]}")

### Un error que no se reintenta

In [ ]:
try:
    get_con_reintentos(requests.Session(), f"{BASE_PEDIDOS}/v1/pedidos", headers={"X-API-Key": "clave-equivocada"})
except ErrorNoReintentable as e:
    print("Falló de inmediato, sin gastar reintentos:", e)

### El cliente robusto

`pedidos()` usa `get_con_reintentos()` (ábrelo en la celda de `cliente_http.py`): timeout de conexión y de lectura,
backoff exponencial con *jitter*, respeta `Retry-After` y guarda un **checkpoint** del cursor después de cada página.

> **Es normal ver líneas `"nivel": "WARNING", "evento": "http_reintento"`**: no son errores, es el cliente haciendo su trabajo.
> La celda tarda entre 10 y 30 segundos y termina mostrando **500 pedidos**. Si se interrumpe, vuelve a correrla:
> continúa desde el checkpoint.

In [ ]:
try:
    df_pedidos = pedidos(requests.Session(), BASE_PEDIDOS, API_KEY, LANDING, FECHA_CARGA, limite=50, timeout=(3.05, 3))
    print(f"\n{len(df_pedidos)} pedidos, {df_pedidos['id'].nunique()} únicos")
    display(df_pedidos.head())
except ErrorNoReintentable as e:
    print("\nLa API rechazó la solicitud:", e)
    print("Si es 401: corrige el secreto, corre la celda 'Primero, el secreto' y después esta.")
except ReintentosAgotados as e:
    print("\nSe agotaron los reintentos:", e)
    print("Si el motivo es ConnectionError, el servidor se detuvo: vuelve a correr la celda de api_caprichosa.py.")
    print("Luego vuelve a correr esta celda: continuará desde el checkpoint.")

## Parte 4 · Los logs también son datos

Como cada evento es una línea JSON, podemos analizarlos con pandas igual que cualquier otro dataset.

In [ ]:
logs = pd.read_json(LANDING / "_logs" / f"lab_{FECHA_CARGA}.jsonl", lines=True)
print("Eventos por tipo:\n", logs["evento"].value_counts().to_string(), "\n")
reintentos = logs[logs["evento"] == "http_reintento"]
print("Reintentos por motivo:\n", reintentos["motivo"].value_counts().to_string() if len(reintentos) else "ninguno")
print(f"\nTiempo total esperando entre reintentos: {reintentos.get('espera_s', pd.Series(dtype=float)).sum():.1f} s")

In [ ]:
import matplotlib.pyplot as plt
ok = logs[logs["evento"] == "http_ok"]
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(len(ok)), ok["ms"], color="#1864AB")
ax.set_title("Latencia de cada llamada exitosa (ms)", loc="left", fontsize=11)
ax.set_xlabel("llamada"); ax.spines[["top", "right"]].set_visible(False)
plt.show()
pd.DataFrame(Manifiesto(LANDING).entradas())[["fuente", "archivo", "estado", "filas", "rechazadas", "hash"]]

## Retos

### Reto 1 · Idempotencia
Vuelve a ingestar los tres archivos. ¿Qué dice el log? ¿Cuántos registros nuevos hay en bronze?

In [ ]:
for fuente, contrato in CONTRATOS.items():
    aterrizar_archivo(ENTRADA / f"proveedor_{fuente}.csv", fuente, contrato, LANDING, FECHA_CARGA)
print("Filas en bronze:", len(pd.read_parquet(LANDING / "bronze" / "inventario")))

### Reto 2 · Un proveedor nuevo
Llegó `proveedor_oriente.csv`. Usa `detectar()` y los primeros bytes del archivo para escribir su contrato.

In [ ]:
print(detectar(ENTRADA / "proveedor_oriente.csv"))

CONTRATOS["oriente"] = {
    "codificacion": "...",       # TODO
    "sep": "...",                # TODO
    "decimal": ".", "miles": None,
    "formato_fecha": "...",      # TODO: revisa cómo viene la columna "corte"
    "columnas": {},              # TODO: {"item": "sku", ...}
}
# aterrizar_archivo(ENTRADA / "proveedor_oriente.csv", "oriente", CONTRATOS["oriente"], LANDING, FECHA_CARGA)

### Reto 3 · Caos y checkpoint
Subimos la tasa de error al 50 % y dejamos solo 2 intentos por llamada. La ingesta de pedidos debería fallar a la mitad.
Luego bajamos el caos y la volvemos a correr: ¿desde qué página continúa?

In [ ]:
ESTADO_API.tasa_error = 0.5
try:
    pedidos(requests.Session(), BASE_PEDIDOS, API_KEY, LANDING, FECHA_CARGA, limite=50,
            max_intentos=2, base=0.1, timeout=(3.05, 3))
    print("Esta vez no falló; vuelve a correr la celda.")
except ReintentosAgotados as e:
    print("Falló:", e)
    print("Checkpoint guardado:", json.loads((LANDING / "_checkpoints" / f"pedidos_{FECHA_CARGA}.json").read_text()))

In [ ]:
ESTADO_API.tasa_error = 0.15
df_pedidos = pedidos(requests.Session(), BASE_PEDIDOS, API_KEY, LANDING, FECHA_CARGA, limite=50, timeout=(3.05, 3))
print(len(df_pedidos), "pedidos. Busca el evento 'reanudando_desde_checkpoint' en el log de arriba.")

## Cierre

| Hoy en el laboratorio | Principio |
|---|---|
| `raw/` guarda lo que llegó, tal cual | La landing zone es inmutable: siempre puedes reprocesar |
| Contrato por proveedor | Detectar para descubrir, contratar para operar |
| `_cuarentena/` y umbral de rechazo | Ninguna fila se pierde en silencio |
| Hash + manifiesto | Ingesta idempotente |
| 401 falla de inmediato; 429 y 5xx se reintentan | Clasificar errores |
| Checkpoint del cursor | Reanudar sin empezar de cero |
| Logs JSON | Observabilidad desde el primer día |

**Tarea:** corre el **kit local** (`kit_ingesta_1_2.zip`) en tu computadora siguiendo su `README.md`.
Es el mismo código, pero con `config.yaml`, `.env`, logs a archivo y códigos de salida: así se ve fuera de un notebook.

**Próxima sesión (1.3):** ingesta desde bases de datos, cargas incrementales y CDC. Trae **Docker Desktop funcionando**.